# 08 · Running the experiment grid

All six methods answer the benchmark questions on **Claude Haiku 5.5** (pinned in `configs/models.yaml`).
The grid itself is `scripts/run_grid.py`; this notebook shows what it ran and how, without re-running anything.

```bash
uv run python scripts/run_grid.py --dry-run   # cost estimate, nothing sent
uv run python scripts/run_grid.py             # run; finished runs are skipped, so it resumes after a crash
```

In [ ]:
import json, sys
from pathlib import Path

import pandas as pd

ROOT = Path("..")
sys.path.append(str(ROOT / "scripts"))
METHODS = ["rag_bm25", "rag_hybrid", "direct_llm", "rlm_depth0", "rlm_depth1", "rlm_depth2"]
SIZES = ["32k", "128k", "512k", "2m"]
runs = pd.concat(pd.read_json(p, lines=True) for p in (ROOT / "results/runs").glob("*.jsonl"))
runs["size"] = pd.Categorical(runs["size"], SIZES, ordered=True)
runs["method"] = pd.Categorical(runs["method"], METHODS, ordered=True)
qs = pd.read_json(ROOT / "data/processed/questions.jsonl", lines=True)
print(f"{len(runs)} runs, {runs.method.nunique()} methods, {runs.id.nunique()} questions")

## 1. The grid
`configs/experiment.yaml` lists which method runs on which contexts, cheapest first. Two methods run on subsets:
* **Direct LLM** cannot read the 2M context (Haiku's limit is 1M tokens), and at 512K each call costs ~25× a 128K-or-smaller
  call, so 512K gets one question per tier.
* **RLM depth 2** runs on T2/T3 at 32K and 128K only: its recursive sub-calls make it the slowest and most variable method.

In [ ]:
import yaml
from run_grid import plan

grid = yaml.safe_load((ROOT / "configs/experiment.yaml").read_text())["grid"]
planned = pd.DataFrame([{"method": m, "size": q["size"]} for m, q in plan(qs, grid)])
planned.pivot_table(index="method", columns="size", aggfunc="size", fill_value=0).reindex(columns=SIZES, fill_value=0)

## 2. Did everything run?
Every planned (method, question) pair has exactly one logged run in `results/runs/<method>.jsonl`.

In [ ]:
done = runs.groupby(["method", "size"], observed=True).size().unstack(fill_value=0).reindex(columns=SIZES, fill_value=0)
print(f"planned {len(planned)}, logged {len(runs)}")
done

## 3. Estimated vs actual cost per run
The dry run assumes an RLM's sub-calls may read the whole context once. Actual per-run costs show where that held and where it didn't.

In [ ]:
from run_grid import estimate
from rlm_logistics.llm import LLM

meta = json.loads((ROOT / "data/processed/contexts.json").read_text())
llm = LLM.from_config("final")
est = pd.DataFrame([{"method": m, "estimate": estimate(m, q, meta[q["size"]]["tokens"], llm)} for m, q in plan(qs, grid)])
pd.DataFrame({"estimated $/run": est.groupby("method").estimate.mean(),
              "actual mean $/run": runs.groupby("method", observed=True).cost.mean(),
              "actual max $/run": runs.groupby("method", observed=True).cost.max()}).reindex(METHODS).round(4)

## 4. What happened while it ran
* **Interruptions.** The run stopped twice (a network outage and a closed session). Because finished runs are logged
  one line at a time and skipped on restart, nothing was lost; runs that hit a connection error were simply retried.
* **One change of order, not of rules.** Depth-2 runs turned out far more expensive than estimated, so the 3 direct-LLM
  runs at 512K were moved ahead of the remaining depth-2 runs to make sure they completed. No method's settings changed.
* **Per-question cost cap.** Every RLM run stops its sub-calls past `budget_usd_per_run` in `configs/experiment.yaml`;
  the most expensive depth-2 runs ended close to it.